# Tutorial 7 — Adding New Stars and Grain Materials

pyGrater ships with two helper modules for extending the built-in catalogs
without editing the text files by hand:

| Module | Target file |
|--------|-------------|
| `pyGrater.add_stars` | `data/star_data/stars_main_properties.txt` |
| `pyGrater.add_materials` | `data/material_list.txt` |

Both modules can also be run directly from the command line.

In [ ]:
import pyGrater
from pyGrater.add_stars import add_star
from pyGrater.add_materials import add_material

# The data path is resolved automatically inside each helper.
# Make sure it is configured before calling them:
print("Data path:", pyGrater.get_data_path())


## 1. Adding a new star

### Required parameters

| Parameter | Unit | Description |
|-----------|------|-------------|
| `star`  | —   | Unique name (no spaces) |
| `dist`  | pc  | Distance |
| `temp`  | K   | Effective temperature |
| `rad`   | R☉  | Stellar radius |
| `logg`  | cgs | Surface gravity log(g) |
| `band`  | —   | Photometric band for normalisation (e.g. `'V'`) |
| `apmag` | mag | Apparent magnitude in that band |

### Optional parameters
`mass`, `spt`, `vsini`, `mdot`, `vw`, `tcoro`, `B0`, `r0`, `tilt`, `per` — all default to `nan`.

In [ ]:
# Minimal example — only required fields
add_star(
    star  = "TutorialStar",
    dist  = 42.0,
    temp  = 6500,
    rad   = 1.3,
    logg  = 4.1,
    band  = "V",
    apmag = 6.2,
)

In [ ]:
# With optional fields
add_star(
    star  = "TutorialStarFull",
    dist  = 42.0,
    temp  = 6500,
    rad   = 1.3,
    logg  = 4.1,
    band  = "V",
    apmag = 6.2,
    mass  = 1.2,       # M_sun
    spt   = "F5V",     # spectral type
    vsini = 15.0,      # km/s
    mdot  = 2.0,       # 10^-14 M_sun/yr
)

In [ ]:
# Verify the new entries can be loaded
from pyGrater import Star

star = Star(star_name="TutorialStarFull")
print(f"Temperature : {star.temp:.0f} K")
print(f"Distance    : {star.distance:.1f} pc")
print(f"Luminosity  : {star.lum:.2f} L_sun")

## 2. Adding a new grain material

> **Before registering a new material**, copy its optical index file(s) into:
> ```
> <data_path>/optical_properties/
> ```
> The filenames passed to `file_par`, `file_per1`, and `file_per2` are resolved
> relative to that folder.

### Required parameters

| Parameter  | Unit   | Description |
|------------|--------|-------------|
| `nickname` | —      | Unique short identifier (no spaces) |
| `Tsub`     | K      | Sublimation temperature |
| `density`  | g/cm³  | Bulk density |
| `file_par` | —      | Optical-property file (parallel orientation) |

If `file_per1` and `file_per2` are **not** provided, all three orientations use `file_par`.

### Optional parameters
`wav_min`, `wav_max`, `weight_par/per1/per2` (default 1/3 each), `full_name`, `formula`,
`material_class`, `subclass`, `group`, `reference`, `web`.


In [ ]:
# Minimal — single optical file reused for all orientations
add_material(
    nickname = "tutorial_dust",
    Tsub     = 1700,
    density  = 3.5,
    file_par = "aC_ACAR.txt",   # reuse an existing file for demonstration
)

In [ ]:
# Full example — separate orientation files + metadata
add_material(
    nickname       = "tutorial_dust_full",
    Tsub           = 1700,
    density        = 3.5,
    file_par       = "c_olivine_Fe_Poor_par.txt",
    file_per1      = "c_olivine_Fe_Poor_per1.txt",
    file_per2      = "c_olivine_Fe_Poor_per2.txt",
    wav_min        = 0.32,
    wav_max        = 4097.0,
    full_name      = "Tutorial olivine (Fe-poor)",
    formula        = "Mg1.9Fe0.1SiO4",
    material_class = "Silicates",
    group          = "Olivine",
    reference      = "Tutorial et al. 2026",
)

In [ ]:
# Verify the new material can be loaded via Grain
from pyGrater import Grain

grain = Grain(composition="tutorial_dust_full")
print(f"Sublimation temperature : {grain.Tsub:.0f} K")
print(f"Qabs shape              : {grain.Qabs.shape}  (N_sizes x N_waves)")
print(f"Wavelength range        : {grain.Qabs_waves.min():.3f} – {grain.Qabs_waves.max():.1f} µm")

## 3. Command-line equivalents

Both scripts can be run from a terminal — useful in batch workflows or when
pyGrater is not yet imported in a Python session.

**Add a star:**
```bash
python -m pyGrater.add_stars \
  --star MyStar --dist 42.0 --temp 6500 --rad 1.3 --logg 4.1 \
  --band V --apmag 6.2 --spt F5V --mass 1.2
```

**Add a material:**
```bash
python -m pyGrater.add_materials \
  --nickname my_dust --Tsub 1700 --density 3.5 \
  --file_par my_dust.txt --full_name "My silicate" --formula MgSiO3
```

Use `--help` on either command to see all available options with descriptions
taken directly from the file headers.

## 4. Summary

| Task | Function | CLI module |
|------|----------|------------|
| Add a star to the catalog | `add_stars.add_star(path, star, dist, temp, rad, logg, band, apmag, ...)` | `python -m pyGrater.add_stars` |
| Add a grain material | `add_materials.add_material(path, nickname, Tsub, density, file_par, ...)` | `python -m pyGrater.add_materials` |

- Both functions raise `ValueError` if the name already exists in the catalog.
- New entries are appended to the file and immediately available in the same Python session.